# WaterVision AI: provenance and licensing

## Dataset
Rudy, I. M., & Wilson, M. J. (2024). *Underwater Image/Turbidity Dataset* (1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10951021. The dataset record declares AGPL-3.0-or-later. The related article is Rudy & Wilson (2024), *PeerJ* 12:e18254, https://doi.org/10.7717/peerj.18254, licensed CC BY 4.0. The article and dataset have distinct license records.

This workflow selects 662 images with usable FNU labels from the upstream Turbidivision v1.0.0 release. Project bands are low (<5 FNU), moderate (5 to <20 FNU), and high (≥20 FNU). Preprocessing converts images to RGB, applies EXIF orientation, resizes within 1024 × 1024 pixels, and saves JPEG at quality 92. Raw images are not bundled in this repository.

## Model and experiments
`yolo26n_full_v1` is the Experiment 01 full-run model. The source notebook configures pretrained `yolo26n-cls.pt`, maximum 50 epochs, 320 input size, batch 16, SGD with initial learning rate 0.005 and cosine schedule, weight decay 0.0005, patience 10, seed 42, deterministic mode, AMP disabled, automatic augmentation disabled, and erasing 0.0.

Experiment 02 consists of three separate family-aware fold models. Its pooled metrics describe those fold models and are not metrics for the Experiment 01 ONNX artifact.

## License
The project uses Ultralytics YOLO and is distributed under AGPL-3.0. The dataset record declares AGPL-3.0-or-later. Ultralytics documents AGPL-3.0 as the default license for its trained models, with Enterprise licensing as an alternative. The repository does not claim an Enterprise license. See `LICENSE`, `docs/licensing.md`, and `docs/dataset.md`.

# WaterVision AI: YOLO26n-cls classification workflow. This notebook demonstrates environment setup, data layout validation, a generic training run, and inference. It is a compact workflow scaffold rather than the complete source experiment notebook.

In [ ]:
%pip install -q ultralytics scikit-learn pandas matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/WaterVisionAI')
DATA_DIR = Path('/content/watervision_data')  # set to authorized train/<class>, val/<class> folders
print('Project folder:', PROJECT_DIR, 'Dataset ready:', DATA_DIR.exists())

## Dataset directory checks
This cell verifies the expected train/validation folder layout. Family-level split integrity is defined by the source manifest and is not established by directory presence alone.

In [ ]:
for split in ('train', 'val'):
    path = DATA_DIR / split
    assert path.is_dir(), f'Missing {path}'
    print(split, [p.name for p in path.iterdir() if p.is_dir()])

## Demonstration training run
This generic single-run configuration is distinct from the reported three-fold family-aware cross-validation experiment.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n-cls.pt')
model.train(data=str(DATA_DIR), epochs=50, imgsz=224, batch=32, device=0, name='watervision_demo')

## Inference with a named checkpoint
For cross-validation work, each fold checkpoint represents a separate training/evaluation split. The fold models are not a single pooled model.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
WEIGHTS = Path('/content/runs/classify/watervision_demo/weights/best.pt')
IMAGE = Path('/content/example.jpg')  # change to an authorized image
assert WEIGHTS.is_file() and IMAGE.is_file()
result = YOLO(str(WEIGHTS)).predict(source=str(IMAGE), imgsz=224)[0]
for i in result.probs.data.argsort(descending=True).tolist():
    print(result.names[i], float(result.probs.data[i]))